In [1]:
print(1)

1


In [2]:
import scanpy as sc
import anndata as ad

In [3]:
import pandas as pd

In [4]:
def create_perturbation_label(is_control: bool, 
                              pert: str, 
                              dose: float, 
                              time: float,
                              well: str,
                              plate: str,
                              design_param: str) -> str:
    if is_control:
        return str(pert) \
                + '_' + str(time) + 'h'
    else:
        if design_param == 'group_all_replicates':
            return str(pert) \
                + '_' + str(dose) \
                + 'uM_' + str(time) + 'h'
        elif design_param == 'separate_replicates':
            return str(pert) \
                + '_' + str(dose) \
                + 'uM_' + str(time) + 'h' \
                + '_' + str(well) \
                + '_' + str(plate)

In [ ]:
def create_dir_if_not_exists(file_output: str) -> None:
    dir_output = os.path.dirname(file_output)
    if not os.path.exists(dir_output):
        try:
            os.makedirs(dir_output)
        except FileExistsError as e:
            logger.warning('%s', str(e))

In [6]:
def add_perturbation_label_to_padata(file_input: str,
                                     file_output: str,
                                     design_param: str,
                                     ) -> None:
    padata = ad.read_h5ad(file_input)
    obs = padata.obs.copy()
    obs['pert_dose_uM'] = obs['pert_dose_uM'].apply(lambda x: format(x, ".15g"))
    obs['pert_time_h'] = obs['pert_time_h'].apply(lambda x: format(x, ".15g"))
    
    padata.obs['perturbation_label'] =  obs.apply(lambda x: create_perturbation_label(x.is_control,
                                                   x.perturbagen,
                                                   x.pert_dose_uM,
                                                   x.pert_time_h,
                                                   x.plate,
                                                   design_param), axis=1).astype("category")
    
    create_dir_if_not_exists(file_output)
    padata.write_h5ad(file_output, compression='gzip')

### sep replicates

In [22]:
padata_sep = sc.read_h5ad('../../data/sciplex/pseudobulk/full/srivatsan20_sciplex3.h5ad')

In [58]:
padata_sep.obs[(padata_sep.obs['plate'] == 'plate52')]

,plate,well,cell_type,perturbagen,pert_type,is_control,pert_dose_uM,pert_time_h,suspension_type,tissue,...,dataset,assay,development_stage,organism,sex,self_reported_ethnicity,pubchem_cid,psbulk_cells,psbulk_counts,perturbation_label
sample_id,,,,,,,,,,,,,,,,,,,,,
plate52_plate10_A10_tazemetostat_CVCL_0023,plate52,plate10_A10,CVCL_0023,tazemetostat,compound,False,10.00,72.0,nucleus,lung,...,srivatsan20_sciplex3,sci-Plex,unknown,human,male,European,66558664,113,224390,tazemetostat_10uM_72h_plate10_A10_plate52
plate52_plate10_A11_JNJ-7706621_CVCL_0023,plate52,plate10_A11,CVCL_0023,JNJ-7706621,compound,False,10.00,72.0,nucleus,lung,...,srivatsan20_sciplex3,sci-Plex,unknown,human,male,European,5330790,3,6189,JNJ-7706621_10uM_72h_plate10_A11_plate52
plate52_plate10_A12_ofloxacin_CVCL_0023,plate52,plate10_A12,CVCL_0023,ofloxacin,compound,False,10.00,72.0,nucleus,lung,...,srivatsan20_sciplex3,sci-Plex,unknown,human,male,European,4583,114,182274,ofloxacin_10uM_72h_plate10_A12_plate52
plate52_plate10_A1_SRT2104_CVCL_0023,plate52,plate10_A1,CVCL_0023,SRT2104,compound,False,0.01,72.0,nucleus,lung,...,srivatsan20_sciplex3,sci-Plex,unknown,human,male,European,25108829,110,191843,SRT2104_0.01uM_72h_plate10_A1_plate52
plate52_plate10_A2_belinostat_CVCL_0023,plate52,plate10_A2,CVCL_0023,belinostat,compound,False,0.01,72.0,nucleus,lung,...,srivatsan20_sciplex3,sci-Plex,unknown,human,male,European,6918638,118,217903,belinostat_0.01uM_72h_plate10_A2_plate52
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
plate52_plate10_H5_PHA-680632_CVCL_0023,plate52,plate10_H5,CVCL_0023,PHA-680632,compound,False,0.01,72.0,nucleus,lung,...,srivatsan20_sciplex3,sci-Plex,unknown,human,male,European,11249084,122,223167,PHA-680632_0.01uM_72h_plate10_H5_plate52
plate52_plate10_H6_PCI-34051_CVCL_0023,plate52,plate10_H6,CVCL_0023,PCI-34051,compound,False,0.01,72.0,nucleus,lung,...,srivatsan20_sciplex3,sci-Plex,unknown,human,male,European,24753719,104,188231,PCI-34051_0.01uM_72h_plate10_H6_plate52
plate52_plate10_H7_ITSA-1_CVCL_0023,plate52,plate10_H7,CVCL_0023,ITSA-1,compound,False,10.00,72.0,nucleus,lung,...,srivatsan20_sciplex3,sci-Plex,unknown,human,male,European,771910,97,148132,ITSA-1_10uM_72h_plate10_H7_plate52


In [23]:
obs = padata_sep.obs.copy()
obs['pert_dose_uM'] = obs['pert_dose_uM'].apply(lambda x: format(x, ".15g"))
obs['pert_time_h'] = obs['pert_time_h'].apply(lambda x: format(x, ".15g"))

padata_sep.obs['perturbation_label'] =  obs.apply(lambda x: create_perturbation_label(x.is_control,
                                               x.perturbagen,
                                               x.pert_dose_uM,
                                               x.pert_time_h,
                                               x.well,
                                               x.plate,
                                               'separate_replicates'), axis=1).astype("category")

In [32]:
len(padata_sep.obs['perturbation_label'].unique())

4872

In [40]:
padata_sep.obs[padata_sep.obs['is_control']==True]['perturbation_label'].unique()

['DMSO_24h', 'DMSO_72h']
Categories (4872, object): ['(+)-JQ1_0.01uM_24h_plate4_E1_plate26', '(+)-JQ1_0.01uM_24h_plate4_E1_plate34', '(+)-JQ1_0.01uM_24h_plate4_E1_plate42', '(+)-JQ1_0.01uM_24h_plate4_E7_plate10', ..., 'zileuton_1uM_24h_plate9_G11_plate7', 'zileuton_1uM_24h_plate9_G5_plate31', 'zileuton_1uM_24h_plate9_G5_plate39', 'zileuton_1uM_24h_plate9_G5_plate47']

In [42]:
len(padata_sep.obs[padata_sep.obs['is_control']!=True]['perturbation_label'])

4870

In [25]:
padata_sep.write_h5ad('../../data/sciplex/pseudobulk_processed/srivatsan20_sciplex3_sep_rep.h5ad')

### group all replicates

In [26]:
padata_group = sc.read_h5ad('../../data/sciplex/pseudobulk/full/srivatsan20_sciplex3.h5ad')

In [27]:
obs = padata_group.obs.copy()
obs['pert_dose_uM'] = obs['pert_dose_uM'].apply(lambda x: format(x, ".15g"))
obs['pert_time_h'] = obs['pert_time_h'].apply(lambda x: format(x, ".15g"))

padata_group.obs['perturbation_label'] =  obs.apply(lambda x: create_perturbation_label(x.is_control,
                                               x.perturbagen,
                                               x.pert_dose_uM,
                                               x.pert_time_h,
                                               x.well,
                                               x.plate,
                                               'group_all_replicates'), axis=1).astype("category")

In [28]:
padata_group.write_h5ad('../../data/sciplex/pseudobulk_processed/srivatsan20_sciplex3_group_rep.h5ad')

In [30]:
padata_group.obs

,plate,well,cell_type,perturbagen,pert_type,is_control,pert_dose_uM,pert_time_h,suspension_type,tissue,...,dataset,assay,development_stage,organism,sex,self_reported_ethnicity,pubchem_cid,psbulk_cells,psbulk_counts,perturbation_label
sample_id,,,,,,,,,,,,,,,,,,,,,
plate10_plate4_A10_tazemetostat_CVCL_0023,plate10,plate4_A10,CVCL_0023,tazemetostat,compound,False,0.01,24.0,nucleus,lung,...,srivatsan20_sciplex3,sci-Plex,unknown,human,male,European,66558664,7,8687,tazemetostat_0.01uM_24h
plate10_plate4_A11_JNJ-7706621_CVCL_0023,plate10,plate4_A11,CVCL_0023,JNJ-7706621,compound,False,0.01,24.0,nucleus,lung,...,srivatsan20_sciplex3,sci-Plex,unknown,human,male,European,5330790,12,12506,JNJ-7706621_0.01uM_24h
plate10_plate4_A12_ofloxacin_CVCL_0023,plate10,plate4_A12,CVCL_0023,ofloxacin,compound,False,0.01,24.0,nucleus,lung,...,srivatsan20_sciplex3,sci-Plex,unknown,human,male,European,4583,9,15805,ofloxacin_0.01uM_24h
plate10_plate4_A1_SRT2104_CVCL_0023,plate10,plate4_A1,CVCL_0023,SRT2104,compound,False,0.10,24.0,nucleus,lung,...,srivatsan20_sciplex3,sci-Plex,unknown,human,male,European,25108829,14,18855,SRT2104_0.1uM_24h
plate10_plate4_A2_belinostat_CVCL_0023,plate10,plate4_A2,CVCL_0023,belinostat,compound,False,0.10,24.0,nucleus,lung,...,srivatsan20_sciplex3,sci-Plex,unknown,human,male,European,6918638,25,33161,belinostat_0.1uM_24h
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
plate9_plate3_H5_PHA-680632_CVCL_0023,plate9,plate3_H5,CVCL_0023,PHA-680632,compound,False,10.00,24.0,nucleus,lung,...,srivatsan20_sciplex3,sci-Plex,unknown,human,male,European,11249084,5,7444,PHA-680632_10uM_24h
plate9_plate3_H6_PCI-34051_CVCL_0023,plate9,plate3_H6,CVCL_0023,PCI-34051,compound,False,10.00,24.0,nucleus,lung,...,srivatsan20_sciplex3,sci-Plex,unknown,human,male,European,24753719,7,10609,PCI-34051_10uM_24h
plate9_plate3_H7_ITSA-1_CVCL_0023,plate9,plate3_H7,CVCL_0023,ITSA-1,compound,False,1.00,24.0,nucleus,lung,...,srivatsan20_sciplex3,sci-Plex,unknown,human,male,European,771910,10,18439,ITSA-1_1uM_24h


In [44]:
len(padata_group.obs['perturbation_label'].unique())

938

In [45]:
padata_group.obs[padata_group.obs['is_control']==True]['perturbation_label'].unique()

['DMSO_24h', 'DMSO_72h']
Categories (938, object): ['(+)-JQ1_0.01uM_24h', '(+)-JQ1_0.01uM_72h', '(+)-JQ1_0.1uM_24h', '(+)-JQ1_0.1uM_72h', ..., 'zileuton_0.01uM_24h', 'zileuton_0.1uM_24h', 'zileuton_10uM_24h', 'zileuton_1uM_24h']

In [50]:
len(padata_group.obs[padata_group.obs['is_control']!=True]['perturbation_label'])

4870

In [ ]:
#padata.write_h5ad('../../data/sciplex/pseudobulk_processed/srivatsan20_sciplex3_group_rep.h5ad')